In [ ]:
import numpy as np
import astropy.io.fits as fits
from astropy.coordinates import SkyCoord, ICRS, Galactic
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.wcs import WCS
from matplotlib import cm
import reproject

In [ ]:
dir_dragons = '/srv/data/dva/dragons_hoard/mini_survey_RFIremoved/610MHz/high/'

hdu_dragons_q = fits.open(dir_dragons+'After_Ground_Iono_610MHz_high_23.q.fits')
hdu_dragons_u = fits.open(dir_dragons+'After_Ground_Iono_610MHz_high_23.u.fits')
hdu_dragons_i = fits.open(dir_dragons+'After_Ground_Iono_610MHz_high_23.i.fits')
dragons_q   = hdu_dragons_q[0].data
dragons_u   = hdu_dragons_u[0].data
dragons_i   = hdu_dragons_i[0].data
hdr_dragons = hdu_dragons_q[0].header


In [ ]:
dir_chime = '/srv/data/chime/chime_QU_Oct2023_400_729/'

hdu_chime_q = fits.open(dir_chime+'Q_400_729_Oct2023_new_eq.fits')
hdu_chime_u = fits.open(dir_chime+'U_400_729_Oct2023_new_eq.fits')
hdu_chime_i = fits.open(dir_chime+'I_400_729_Oct2023_new.fits')
hdr_chime3D = hdu_chime_q[0].header
#print(repr(hdr_chime3D))

freq = WCS(hdr_chime3D).all_pix2world(0,0,range(hdu_chime_q[0].data.shape[0]),0)[2]
fidx = np.where(abs(freq-610e6) == np.min(abs(freq-610e6)))[0][0]
print(freq[fidx])

In [ ]:
chime_q = hdu_chime_q[0].data[fidx]
chime_u = hdu_chime_u[0].data[fidx]
chime_i = hdu_chime_i[0].data[fidx]

hdr_chime = hdr_chime3D.copy()
del hdr_chime['NAXIS3']
del hdr_chime['CTYPE3']
del hdr_chime['CRPIX3']
del hdr_chime['CRVAL3']
del hdr_chime['CDELT3']
del hdr_chime['CUNIT3']
hdr_chime['NAXIS'] = 2
print(repr(hdr_chime))

In [ ]:
def make_plots_eq(data, hdr, RA_range=[24,0], dec_range=[0,90], 
                             vmin=-40, vmax=40, cmapuse = 'RdBu_r',cbarlbl='Jy/beam'):

    fs = 16
    crange = SkyCoord(RA_range, dec_range, frame=ICRS, unit=(u.hourangle, u.deg))

    fig = plt.figure(figsize=(18,6))
    ax = fig.add_subplot(111, projection=WCS(hdr).celestial)

    cmap = cm.get_cmap(cmapuse).copy()
    cmap.set_bad('lightgray')

    im = ax.imshow(data,origin='lower',vmin=vmin,vmax=vmax,cmap=cmap)

    ax.set_xlim(WCS(hdr).world_to_pixel(crange)[0])
    ax.set_ylim(WCS(hdr).world_to_pixel(crange)[1])

    cbar = fig.colorbar(im,ax=ax,orientation='vertical',fraction=0.01,pad=0.02,aspect=20)
    cbar.ax.tick_params(axis='y', which='both',labelsize=fs)
    cbar.set_label(cbarlbl, fontsize=fs)

    ax.tick_params(axis='both', which='major', labelsize=fs, zorder=30, length=5)
    ax.coords[0].display_minor_ticks(True)
    ax.coords[1].display_minor_ticks(True)
    ax.set_xlabel('RA',fontsize=fs)
    ax.set_ylabel('dec',fontsize=fs)

    #plt.tight_layout()
    #plt.savefig('/home/aordog/DVA_PLOTS/east_west_Jan03/'+prefix+'_'+freqname+'.pdf')
   
    return

In [ ]:
make_plots_eq(dragons_u, hdr_dragons,vmin=-30,vmax=30)
make_plots_eq(chime_u, hdr_chime,vmin=-2,vmax=2)

In [ ]:
dragons_q_hpx,footprint = reproject.reproject_to_healpix((dragons_q, hdr_dragons),coord_system_out='C',nside=512)
dragons_q_regrid,footprint = reproject.reproject_from_healpix((dragons_q_hpx,'C'),hdr_chime, nested=False)

dragons_u_hpx,footprint = reproject.reproject_to_healpix((dragons_u, hdr_dragons),coord_system_out='C',nside=512)
dragons_u_regrid,footprint = reproject.reproject_from_healpix((dragons_u_hpx,'C'),hdr_chime, nested=False)

In [ ]:
dragons_q_K = dragons_q_regrid/20
dragons_u_K = dragons_u_regrid/20
chime_q_K   = chime_q*1.61
chime_u_K   = chime_u*1.61

In [ ]:
make_plots_eq(dragons_u_K, hdr_chime,vmin=-3,vmax=3,cbarlbl='K')
make_plots_eq(chime_u_K , hdr_chime,vmin=-3,vmax=3,cbarlbl='K')

In [ ]:
chime_q_K[np.isnan(chime_q_K)] = 0
chime_u_K[np.isnan(chime_u_K)] = 0
dragons_q_K[np.isnan(dragons_q_K)] = 0
dragons_u_K[np.isnan(dragons_u_K)] = 0

In [ ]:
def feather_ST(image,dxy,R1=12.9,R2=17.1,*arg,**kwargs):
        
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq_m*(3e8/610e6)*180/(np.pi**2)
    uv_n = uv_freq_n*(3e8/610e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_n)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask = 0.5*np.sin(b*(ruv - r0))+0.5
    mask[ruv > R2] = 1
    mask[ruv < R1] = 0
    
    image_FFT_shift = image_FFT_shift*mask
    image_ST = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_ST, mask, uv_m, uv_n

def feather_SA(image,dxy,R1=12.9,R2=17.1,*arg,**kwargs):

    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq_m*(3e8/610e6)*180/(np.pi**2)
    uv_n = uv_freq_n*(3e8/610e6)*180/(np.pi**2)
    #print(uv_m) # Units of 1/deg

    xuv, yuv = np.meshgrid(uv_m, uv_n)
    ruv = np.sqrt(xuv**2+yuv**2)

    #extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    b = np.pi/(R2-R1)
    r0 = (R1+R2)/2.
    mask = -0.5*np.sin(b*(ruv - r0))+0.5
    mask[ruv > R2] = 0
    mask[ruv < R1] = 1
    
    image_FFT_shift = image_FFT_shift*mask
    image_SA = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
       
    return image_SA, mask, uv_m, uv_n


def SA_deconvolve2(image_full, SA_beam, noiseslope, plots=True, *args,**kwargs):
    
    image_full_FFT = np.fft.fft2(image_full)
    image_full_FFT_shift = np.fft.fftshift(image_full_FFT)

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq_m*(3e8/610e6)*180/(np.pi**2)
    uv_n = uv_freq_n*(3e8/610e6)*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_n[0],uv_n[-1]]
    
    noise = noiseslope*np.random.normal(size=([image_full.shape[0],image_full.shape[1]]))
   
    if plots:
        # Plot simulated observation (SA image):
        fig,ax = plt.subplots(2,2,figsize=(12,10))
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image_full.real,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_full_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    image_FFT_shift = np.empty_like(image_full_FFT_shift)
    image_FFT_shift[SA_beam>1e-300] = image_full_FFT_shift[SA_beam>1e-300]#+noise[SA_beam>1e-300]/SA_beam[SA_beam>1e-300]
    #image_FFT_shift[SA_beam<1e-300] = 1e300
    print('Threshold for zero: ',np.min(ruv[SA_beam<1e-300]))
    
    if plots:
        # Plot deconvolved:
        ax[1,1].set_title('SA beam deconvolved')
        ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
        ax[1,1].plot(uv_m,np.log10(abs(image_FFT_shift[511,:])))
        ax[1,1].plot(uv_m,np.log10(abs(image_full_FFT_shift[511,:])))
        ax[1,1].set_ylim(0,10),ax[1,1].set_xlim(-40,40),ax[1,1].grid()
     
    return image_FFT_shift

def get_beam(image,dxy,pix0,R=9,plots=True,*arg,**kwargs):
    
    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy
    uv_m = uv_freq*(3e8/1420e6)*180/(np.pi**2)

    xuv, yuv = np.meshgrid(uv_m, uv_m)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    extent=[uv_m[0],uv_m[-1],uv_m[0],uv_m[-1]]
    
    mask = gf(0,2*R,ruv)
    
    if plots:
        # Plot original image:
        fig,ax = plt.subplots(2,2,figsize=(12,10))
        ax[1,0].set_title('Original image')
        ax[1,0].imshow(image,origin='lower',vmin=0,vmax=30)
        ax[0,0].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)
    
    # Apply beam mask and plot:
    image_FFT_shift = image_FFT_shift*mask
    image_SA_nonoise = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
    
    if plots:
        ax[1,1].set_title('Beam mask applied')
        ax[1,1].imshow(image_SA_nonoise.real,origin='lower',vmin=0,vmax=30)
        ax[0,1].imshow(abs(image_FFT_shift),origin='lower',vmin=0,vmax=1e5,extent=extent)   
    
    return mask, ruv[pix0,:],mask[pix0,:],image_SA_nonoise.real

In [ ]:
pix0y = hdr_chime['CRPIX2']
pix0x = hdr_chime['CRPIX1']
print(pix0x,pix0y)

In [ ]:
dxy = np.round(hdr_chime['CDELT2'],5)
print(dxy)

In [ ]:
chime_HP, mask_HP, uv_m, uv_n = feather_ST(chime_q_K+chime_u_K*1j,dxy,R1=1,R2=4)
dragons_LP, mask_LP, uv_m, uv_n = feather_SA(dragons_q_K+dragons_u_K*1j,dxy,R1=1,R2=4)
#plt.imshow(mask_HP)
plt.imshow(mask_LP)

In [ ]:
make_plots_eq(dragons_u_K , hdr_chime,vmin=-2,vmax=2,cbarlbl='K',RA_range=[24,0], dec_range=[15,90])
make_plots_eq(chime_u_K , hdr_chime,vmin=-2,vmax=2,cbarlbl='K',RA_range=[24,0], dec_range=[15,90])
make_plots_eq(chime_HP.imag+dragons_LP.imag , hdr_chime,vmin=-2,vmax=2,cbarlbl='K',RA_range=[24,0], dec_range=[15,90])

In [ ]:
make_plots_eq(chime_u_K , hdr_chime,vmin=-3,vmax=3,cbarlbl='K')
make_plots_eq(chime_u_IFFT , hdr_chime,vmin=-3,vmax=3,cbarlbl='K')